# BERT Walkthrough: Seeing the Transformer Encoder in Action

In today's lecture, we learned:
- Why static embeddings (Word2Vec, GloVe) aren't enough
- How BERT uses the **encoder side** of the Transformer to create **contextual embeddings**
- How BERT was trained with **Masked Language Modeling** and **Next Sentence Prediction**

Now let's see it all **live**. We'll:
1. **Set up** BERT and check our GPU
2. **Tokenize text** and see WordPiece in action
3. **Extract contextual embeddings** and prove the same word gets different vectors
4. **Play fill-in-the-blank** with BERT's masked language modeling
5. **Run sentiment analysis** using a fine-tuned BERT model
6. **Compare sentence similarity** using BERT embeddings

Let's go! 

---
## Part 1: Setup & First Look

**Goal:** Install libraries, check GPU, and see how BERT tokenizes text.

Think of this like opening the hood of a car — we're going to look at all the parts before we start driving.

In [1]:
# Install the libraries we need (Colab already has PyTorch!)
# !pip install -q transformers

In [2]:
# Check if we have a GPU available
# In Colab: Runtime > Change runtime type > GPU
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    device = torch.device("cpu")
    print("No GPU found — using CPU (things will be slower)")
    print("Go to Runtime > Change runtime type > GPU to fix this!")

No GPU found — using CPU (things will be slower)
Go to Runtime > Change runtime type > GPU to fix this!


### BERT's Tokenizer: WordPiece in Action

Remember from the slides: BERT doesn't use whole words. It uses **WordPiece** tokenization, which breaks rare or complex words into smaller pieces.

Think of it like Lego blocks — instead of having a unique brick for every possible shape, you have a set of common pieces that can be combined to build anything.

In [3]:
from transformers import BertTokenizer

# Load BERT's tokenizer (this is the same one used during pre-training)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

print(f"Vocabulary size: {tokenizer.vocab_size:,} tokens")
print(f"That's {tokenizer.vocab_size:,} 'Lego pieces' BERT knows!")

/Users/davidfekke/Documents/python/projects/mynotebooks/unit-5-lessons/.pixi/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/davidfekke/Documents/python/projects/mynotebooks/unit-5-lessons/.pixi/envs/default/lib/python3.12/site-packages/sympy/external/gmpy.py:139: UserWarning: gmpy2 version is too old to use (2.0.0 or newer required)
  gmpy = import_module('gmpy2', min_module_version=_GMPY2_MIN_VERSION,


Vocabulary size: 30,522 tokens
That's 30,522 'Lego pieces' BERT knows!


In [4]:
# Let's see how BERT tokenizes a sentence
sentence = "I love machine learning and artificial intelligence!"

# Tokenize
tokens = tokenizer.tokenize(sentence)
print(f"Original:  {sentence}")
print(f"Tokens:    {tokens}")
print(f"# tokens:  {len(tokens)}")

# Now let's try a word BERT probably hasn't seen as a whole word
sentence2 = "Transformers use self-attention mechanisms for contextualization."
tokens2 = tokenizer.tokenize(sentence2)
print(f"\nOriginal:  {sentence2}")
print(f"Tokens:    {tokens2}")
print(f"# tokens:  {len(tokens2)}")
print("\n👆 See the '##' prefix? That means 'this is a continuation of the previous piece'")

Original:  I love machine learning and artificial intelligence!
Tokens:    ['i', 'love', 'machine', 'learning', 'and', 'artificial', 'intelligence', '!']
# tokens:  8

Original:  Transformers use self-attention mechanisms for contextualization.
Tokens:    ['transformers', 'use', 'self', '-', 'attention', 'mechanisms', 'for', 'context', '##ual', '##ization', '.']
# tokens:  11

👆 See the '##' prefix? That means 'this is a continuation of the previous piece'


In [7]:
# Let's see the FULL encoding — with special tokens!
encoded = tokenizer(sentence, return_tensors='pt')

# Decode to see what BERT actually receives
full_tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])
print("What BERT actually sees:")
print(full_tokens)
print(f"\n[CLS] = 'Hey BERT, a new input starts here'")
print(f"[SEP] = 'This is the end of the input'")
print(f"\nThese special tokens are like punctuation marks that help BERT understand the structure.")

What BERT actually sees:
['[CLS]', 'i', 'love', 'machine', 'learning', 'and', 'artificial', 'intelligence', '!', '[SEP]']

[CLS] = 'Hey BERT, a new input starts here'
[SEP] = 'This is the end of the input'

These special tokens are like punctuation marks that help BERT understand the structure.


In [8]:
# YOUR TURN: Change the name below to YOUR name!
my_name = "Abishek"

tokens = tokenizer.tokenize(my_name.lower())
print(f"Name: {my_name}")
print(f"BERT sees: {tokens}")

if len(tokens) > 1:
    print(f"\n🔍 BERT split your name into {len(tokens)} pieces!")
    print("That means your name isn't in BERT's vocabulary as a single token.")
    print("But BERT can still understand it by combining the pieces!")
else:
    print(f"\n Your name is a single token — BERT knows it as one piece!")

Name: Abishek
BERT sees: ['ab', '##ish', '##ek']

🔍 BERT split your name into 3 pieces!
That means your name isn't in BERT's vocabulary as a single token.
But BERT can still understand it by combining the pieces!


---
## Part 2: Contextual Embeddings in Action

**Goal:** Prove that BERT gives the SAME word DIFFERENT embeddings based on context.

This is the big upgrade over Word2Vec! Remember:
- **Word2Vec/GloVe:** "bank" always gets the SAME vector, whether it's a river bank or a money bank
- **BERT:** "bank" gets a DIFFERENT vector depending on the sentence around it

It's like how the word "cool" means different things when your friend says "that's cool!" vs. when the weather report says "cool temperatures tonight."

In [9]:
from transformers import BertModel
import torch.nn.functional as F

# Load the pre-trained BERT model
model = BertModel.from_pretrained('bert-base-uncased')
model.eval()  # Set to evaluation mode (we're not training)
model.to(device)

print(f"BERT loaded! {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters")
print("That's 110 million numbers that BERT learned during pre-training!")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19638.75it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT loaded! 109M parameters
That's 110 million numbers that BERT learned during pre-training!


In [10]:
def get_word_embedding(sentence, target_word):
    """Get BERT's contextual embedding for a specific word in a sentence."""
    # Tokenize
    inputs = tokenizer(sentence, return_tensors='pt').to(device)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    # Find the target word's token position
    target_idx = None
    for i, token in enumerate(tokens):
        if token == target_word.lower():
            target_idx = i
            break

    if target_idx is None:
        print(f"Warning: '{target_word}' not found as a single token in: {tokens}")
        return None

    # Get BERT's output
    with torch.no_grad():
        outputs = model(**inputs)

    # Extract the embedding for our target word
    # outputs.last_hidden_state shape: (batch, seq_len, 768)
    embedding = outputs.last_hidden_state[0, target_idx, :]
    return embedding

In [11]:
# The word "sick" in 3 VERY different contexts
sentences = [
    "I feel sick and need to see a doctor.",       # sick = ill
    "That skateboard trick was sick!",              # sick = awesome
    "I'm sick of waiting in long lines.",           # sick = tired/fed up
]

# Get the embedding for "sick" in each sentence
embeddings = []
for sent in sentences:
    emb = get_word_embedding(sent, "sick")
    embeddings.append(emb)
    print(f"'{sent}'")
    print(f"  → Embedding shape: {emb.shape}")
    print(f"  → First 5 values: {emb[:5].cpu().numpy().round(3)}")
    print()

'I feel sick and need to see a doctor.'
  → Embedding shape: torch.Size([768])
  → First 5 values: [ 0.431  0.828  0.707 -0.59  -0.19 ]

'That skateboard trick was sick!'
  → Embedding shape: torch.Size([768])
  → First 5 values: [ 0.615  0.163  0.197 -0.075 -0.838]

'I'm sick of waiting in long lines.'
  → Embedding shape: torch.Size([768])
  → First 5 values: [ 0.147 -0.21   0.13   0.129 -0.387]



In [12]:
# Now let's MEASURE how different these embeddings are using cosine similarity
# Cosine similarity: 1.0 = identical, 0.0 = completely different

print("Cosine Similarity between 'sick' in different contexts:")
print("=" * 55)

labels = ["ill", "awesome", "fed up"]
for i in range(len(embeddings)):
    for j in range(i + 1, len(embeddings)):
        sim = F.cosine_similarity(embeddings[i].unsqueeze(0), embeddings[j].unsqueeze(0))
        print(f"  '{labels[i]}' vs '{labels[j]}': {sim.item():.4f}")

print("\n💡 KEY INSIGHT: Same word, different numbers!")
print("   Word2Vec would give similarity = 1.0000 for all pairs")
print("   because it only has ONE vector per word.")
print("\n   BERT understands that 'sick' means different things")
print("   depending on the words around it — just like we do!")

Cosine Similarity between 'sick' in different contexts:
  'ill' vs 'awesome': 0.5163
  'ill' vs 'fed up': 0.4904
  'awesome' vs 'fed up': 0.4375

💡 KEY INSIGHT: Same word, different numbers!
   Word2Vec would give similarity = 1.0000 for all pairs
   because it only has ONE vector per word.

   BERT understands that 'sick' means different things
   depending on the words around it — just like we do!


---
## Part 3: BERT's Fill-in-the-Blank Superpower

**Goal:** See Masked Language Modeling (MLM) live — the task BERT was actually trained on!

Remember: BERT learned language by playing fill-in-the-blank millions of times. It would see a sentence with a word hidden behind `[MASK]`, and had to guess the missing word.

It's like the world's most intense game of Mad Libs — except BERT played it on the entire internet.

In [13]:
from transformers import pipeline

# Load the fill-mask pipeline — this uses BERT's MLM head
fill_mask = pipeline("fill-mask", model="bert-base-uncased", device=0 if torch.cuda.is_available() else -1)

print("Fill-mask pipeline loaded! Let's play fill-in-the-blank with BERT.")

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 20270.58it/s]
BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fill-mask pipeline loaded! Let's play fill-in-the-blank with BERT.


In [14]:
# Example 1: An easy one
results = fill_mask("The capital of France is [MASK].")

print("Sentence: 'The capital of France is [MASK].'")
print("\nBERT's top 5 guesses:")
for r in results:
    print(f"  {r['token_str']:>12}  (confidence: {r['score']:.1%})")

Sentence: 'The capital of France is [MASK].'

BERT's top 5 guesses:
         paris  (confidence: 41.7%)
         lille  (confidence: 7.1%)
          lyon  (confidence: 6.3%)
     marseille  (confidence: 4.4%)
         tours  (confidence: 3.0%)


In [23]:
# Example 2: Context changes everything!
# Same structure, different context → different answers

sentences = [
    "I went to the [MASK] to deposit money.",
    "I sat by the river [MASK] and watched the sunset.",
    "The doctor told me to take my [MASK] every morning.",
    "She studied [MASK] at the university for four years in the United States of America.",
]

for sent in sentences:
    results = fill_mask(sent)
    top3 = [f"{r['token_str']} ({r['score']:.1%})" for r in results[:3]]
    print(f"'{sent}'")
    print(f"  → Top 3: {', '.join(top3)}")
    print()

'I went to the [MASK] to deposit money.'
  → Top 3: bank (87.1%), store (2.2%), office (0.9%)

'I sat by the river [MASK] and watched the sunset.'
  → Top 3: ##bank (67.4%), bank (12.2%), , (4.3%)

'The doctor told me to take my [MASK] every morning.'
  → Top 3: medication (39.7%), pill (12.3%), pills (9.3%)

'She studied [MASK] at the university for four years in the United States of America.'
  → Top 3: law (13.8%), english (6.0%), psychology (6.0%)



In [16]:
# YOUR TURN: Write your own [MASK] sentence!
my_sentence = "I love to eat [MASK] for breakfast."  # ← Change this!

results = fill_mask(my_sentence)
print(f"Your sentence: '{my_sentence}'")
print("\nBERT's guesses:")
for r in results:
    print(f"  {r['token_str']:>12}  (confidence: {r['score']:.1%})")

Your sentence: 'I love to eat [MASK] for breakfast.'

BERT's guesses:
            it  (confidence: 13.5%)
          them  (confidence: 8.6%)
           you  (confidence: 7.2%)
     something  (confidence: 6.0%)
           her  (confidence: 5.7%)


### 💡 What just happened?

This fill-in-the-blank game IS how BERT learned language! During pre-training:
1. Take a sentence from the internet
2. Randomly hide 15% of the words behind `[MASK]`
3. Make BERT guess what was hidden
4. If wrong, adjust the weights (backpropagation!)
5. Repeat... **millions** of times

By doing this, BERT learned grammar, facts, word meanings, and even some reasoning — all without anyone labeling data! This is **self-supervised learning**.

---
## Part 4: Sentiment Analysis with BERT

**Goal:** Use a fine-tuned BERT model for a real task — understanding if text is positive or negative.

This is the **pre-train + fine-tune** pattern in action:
1. **Pre-train:** BERT learns language from billions of words (fill-in-the-blank)
2. **Fine-tune:** Someone takes that pre-trained BERT and trains it a little more on labeled sentiment data

It's like how a chef (pre-trained on cooking basics) can quickly learn a new cuisine (fine-tuned for a specific task). They don't start from scratch!

In [17]:
# Load a sentiment analysis pipeline
# This uses a BERT model that was fine-tuned on movie review sentiment data
sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if torch.cuda.is_available() else -1
)

print("Sentiment analysis pipeline loaded!")
print("This model was fine-tuned on the SST-2 dataset (movie reviews).")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 17299.53it/s]


Sentiment analysis pipeline loaded!
This model was fine-tuned on the SST-2 dataset (movie reviews).


In [18]:
# Let's start with some easy ones
easy_sentences = [
    "This movie was absolutely fantastic! I loved every minute of it.",
    "Terrible film. Complete waste of time and money.",
    "The food at this restaurant is amazing.",
    "I'm so disappointed with the customer service.",
]

print("Easy Sentences — Can BERT get these right?")
print("=" * 55)
for sent in easy_sentences:
    result = sentiment(sent)[0]
    emoji = "😊" if result['label'] == 'POSITIVE' else "😞"
    print(f"{emoji} {result['label']:>8} ({result['score']:.1%})  '{sent[:50]}...'" if len(sent) > 50
          else f"{emoji} {result['label']:>8} ({result['score']:.1%})  '{sent}'")

Easy Sentences — Can BERT get these right?
😊 POSITIVE (100.0%)  'This movie was absolutely fantastic! I loved every...'
😞 NEGATIVE (100.0%)  'Terrible film. Complete waste of time and money.'
😊 POSITIVE (100.0%)  'The food at this restaurant is amazing.'
😞 NEGATIVE (100.0%)  'I'm so disappointed with the customer service.'


In [19]:
# Now let's try some TRICKY ones — sarcasm, mixed feelings, subtle language
tricky_sentences = [
    "Oh great, another meeting that could have been an email.",     # sarcasm
    "The movie was not bad, actually pretty decent.",               # double negative
    "I expected it to be terrible but it was surprisingly okay.",   # mixed
    "The special effects were stunning but the plot was garbage.",  # mixed
    "Well, that was certainly... something.",                       # ambiguous
]

print("Tricky Sentences — Can BERT handle nuance?")
print("=" * 55)
for sent in tricky_sentences:
    result = sentiment(sent)[0]
    emoji = "😊" if result['label'] == 'POSITIVE' else "😞"
    conf = "confident" if result['score'] > 0.9 else "unsure" if result['score'] < 0.7 else "moderate"
    print(f"{emoji} {result['label']:>8} ({result['score']:.1%}, {conf})")
    print(f"   '{sent}'")
    print()

Tricky Sentences — Can BERT handle nuance?
😞 NEGATIVE (92.8%, confident)
   'Oh great, another meeting that could have been an email.'

😊 POSITIVE (99.9%, confident)
   'The movie was not bad, actually pretty decent.'

😊 POSITIVE (99.9%, confident)
   'I expected it to be terrible but it was surprisingly okay.'

😞 NEGATIVE (100.0%, confident)
   'The special effects were stunning but the plot was garbage.'

😊 POSITIVE (99.6%, confident)
   'Well, that was certainly... something.'



In [20]:
# Batch analyze a small dataset — like you might in a real project!
reviews = [
    "Absolutely loved this product, exceeded all expectations!",
    "It works fine, nothing special but gets the job done.",
    "Broke after two days. Waste of money. Never buying again.",
    "The packaging was nice but the product itself was mediocre.",
    "Best purchase I've made all year. Highly recommend!",
    "Meh. It's okay I guess. Wouldn't go out of my way to buy it.",
    "DO NOT BUY THIS. Worst experience of my life.",
    "Pretty good for the price. Would buy again.",
]

print("Mini Review Dataset — Batch Analysis")
print("=" * 60)
results = sentiment(reviews)

positive = 0
negative = 0
for review, result in zip(reviews, results):
    emoji = "😊" if result['label'] == 'POSITIVE' else "😞"
    print(f"{emoji} {result['label']:>8} ({result['score']:.1%})  {review[:55]}...")
    if result['label'] == 'POSITIVE':
        positive += 1
    else:
        negative += 1

print(f"\n📊 Summary: {positive} positive, {negative} negative out of {len(reviews)} reviews")
print(f"   Overall sentiment: {'Mostly Positive 👍' if positive > negative else 'Mostly Negative 👎'}")

Mini Review Dataset — Batch Analysis
😊 POSITIVE (100.0%)  Absolutely loved this product, exceeded all expectation...
😊 POSITIVE (99.8%)  It works fine, nothing special but gets the job done....
😞 NEGATIVE (100.0%)  Broke after two days. Waste of money. Never buying agai...
😞 NEGATIVE (99.9%)  The packaging was nice but the product itself was medio...
😊 POSITIVE (100.0%)  Best purchase I've made all year. Highly recommend!...
😞 NEGATIVE (77.6%)  Meh. It's okay I guess. Wouldn't go out of my way to bu...
😞 NEGATIVE (100.0%)  DO NOT BUY THIS. Worst experience of my life....
😊 POSITIVE (100.0%)  Pretty good for the price. Would buy again....

📊 Summary: 4 positive, 4 negative out of 8 reviews
   Overall sentiment: Mostly Negative 👎


---
## Part 5: Sentence Similarity

**Goal:** Use BERT to compare entire sentences — which ones are most similar in meaning?

This is how search engines work! When you type a query, the engine converts your query AND all the documents into vectors, then finds the documents whose vectors are closest to yours.

Think of it like GPS coordinates — two restaurants might have different names and menus, but if their coordinates are close together, they're in the same neighborhood. BERT gives sentences "meaning coordinates."

In [21]:
def get_sentence_embedding(sentence):
    """Get a single vector representing the entire sentence using BERT's [CLS] token."""
    inputs = tokenizer(sentence, return_tensors='pt', padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    # Use the [CLS] token embedding as the sentence representation
    # [CLS] is always the first token — BERT learns to pack sentence-level meaning here
    return outputs.last_hidden_state[0, 0, :]  # shape: (768,)

In [22]:
# Let's compare sentences!
sentences = [
    "The cat sat on the mat.",                    # 0: about a cat
    "A kitten was resting on the rug.",           # 1: similar meaning, different words!
    "The stock market crashed yesterday.",         # 2: completely different topic
    "Dogs are wonderful companions.",              # 3: related (pets) but different
    "Financial markets experienced a downturn.",   # 4: similar to #2, different words
]

# Get embeddings for all sentences
embeddings = [get_sentence_embedding(s) for s in sentences]

# Compute all pairwise similarities
print("Sentence Similarity Matrix")
print("=" * 70)
print(f"\n{'':>4}", end="")
for i in range(len(sentences)):
    print(f"  [{i}]  ", end="")
print()

for i in range(len(sentences)):
    print(f"[{i}] ", end="")
    for j in range(len(sentences)):
        sim = F.cosine_similarity(embeddings[i].unsqueeze(0), embeddings[j].unsqueeze(0)).item()
        print(f" {sim:.3f} ", end="")
    print()

print("\nSentences:")
for i, s in enumerate(sentences):
    print(f"  [{i}] {s}")

Sentence Similarity Matrix

      [0]    [1]    [2]    [3]    [4]  
[0]  1.000  0.897  0.814  0.879  0.728 
[1]  0.897  1.000  0.762  0.790  0.726 
[2]  0.814  0.762  1.000  0.807  0.793 
[3]  0.879  0.790  0.807  1.000  0.721 
[4]  0.728  0.726  0.793  0.721  1.000 

Sentences:
  [0] The cat sat on the mat.
  [1] A kitten was resting on the rug.
  [2] The stock market crashed yesterday.
  [3] Dogs are wonderful companions.
  [4] Financial markets experienced a downturn.


---
## Wrap-Up & Key Takeaways

### What We Just Did:

| Part | What We Saw | Key Concept |
|------|------------|-------------|
| **Tokenization** | BERT breaks text into WordPiece tokens | Subword tokenization handles any word |
| **Contextual Embeddings** | "sick" gets different vectors in different sentences | Self-attention reads the whole sentence |
| **Fill-in-the-Blank** | BERT guesses masked words with high accuracy | This is how BERT was pre-trained (MLM) |
| **Sentiment Analysis** | Classify text as positive/negative with 3 lines of code | Pre-train + Fine-tune = Transfer Learning |
| **Sentence Similarity** | Compare meaning of sentences, build a mini search engine | Semantic search > keyword matching |

### The Big Picture:

```
Pre-training (Google did this)          Fine-tuning (anyone can do this)
┌─────────────────────────┐           ┌─────────────────────────────┐
│ Books + Wikipedia       │           │ + Small labeled dataset     │
│ Masked Language Modeling │    →     │ = Sentiment, Q&A, NER, etc. │
│ Next Sentence Prediction│           │   (just add a output layer) │
└─────────────────────────┘           └─────────────────────────────┘
```

### What's Next:
- **Decoder side:** How GPT generates text (next class!)
- **Attention deep-dive:** The math behind self-attention
- **Building with LLMs:** From BERT's understanding to GPT's generation

**BERT showed the world that pre-training + fine-tuning is the way.** Every major language model since — GPT, T5, LLaMA, Claude — builds on this foundation.